In [74]:
import torch
import torch.nn as nn
import bidirectional_dataset
import plotting
import importlib
import neptune
import linear_benchmark

importlib.reload(bidirectional_dataset)
importlib.reload(plotting)

<module 'plotting' from '/home/dl11e23/sleep-time-series/plotting.py'>

# Setup hyperparameters

In [75]:
run_name = "Benchmark_cross_validation"
# create a new neptune run
run = neptune.init_run(project="sleep-time-series/sleep-time-series",
                        api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJlNjYwOTA1Zi02ZDhiLTQ5NGYtODUwYy1jNzA3ZmM0MjBhMWEifQ==",
                        name=run_name,
                        tags=["Benchmark"])

# Exclude subject 006
subjects = ["001", "002", "003", "004", "005", "007", "008", "009"]


# in seconds
time_before_cutout = 1
cutout_duration = 1
time_after_cutout = 1

# defining the resampling stuff
original_freq = 200
resample_freq = 20

# HYPERPARAMETERS

batch_size = 512

n_channels = 7
cutout_duration_steps = cutout_duration * resample_freq


https://app.neptune.ai/sleep-time-series/sleep-time-series/e/SLEEP-403


## Model

In [76]:
importlib.reload(linear_benchmark)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark_model = linear_benchmark.BenchmarkModel() # Use benchmark model
benchmark_model.to(device)
loss_fn = nn.MSELoss(reduction="mean") # Same loss function as actual model 


In [ ]:
def run_benchmark(dataloader, benchmark_model = benchmark_model, loss_fn=loss_fn, device=device):
    '''
    Predicts target using benchmark model for all batches in a dataloader
    and returns mean loss. 
    Args:
        dataloader: DataLoader to run and evaluate on
        benchmark_model: model to evaluate
        loss_fn: loss function to use in evaluation
        device: device to run model on
    '''
    benchmark_model.eval()
    total_loss = 0
    with torch.no_grad():
        for xbatch,ybatch in dataloader: # loop over all batches
            xbatch[0], xbatch[1], ybatch[0], ybatch[1] = (
                xbatch[0].to(device),
                xbatch[1].to(device),
                ybatch[0].to(device),
                ybatch[1].to(device)
            )
            output = benchmark_model(xbatch, ybatch, 0).to(device)
            total_loss += loss_fn(output, ybatch[0]).item() # Calculate and add batch loss to total loss
    
    # return mean loss
    return total_loss / len(dataloader) 

## Benchmark cross validation loop

In [78]:
%%capture
# this supresses the output

losses = []
n_sections = []
for i in range(len(subjects)): # loop over all subjects
    validation_subjects = [subjects[i]] # one subject is validation data
    training_subjects = subjects[:i] + subjects[i+1:] # all other subjects is training data
    
    # Create training dataset to get standardization info
    _, _, standardization_info = bidirectional_dataset.create_combined_dataset(subjects=training_subjects,
                                                                                proportion_to_use=1, time_before_cutout=time_before_cutout,
                                                                                                 cutout_duration=cutout_duration,
                                                                                original_freq=original_freq, resample_freq=resample_freq)

    # Create a combined and individual validation dataset using standardization from the training dataset
    combined_validation_dataset, individual_validation_datasets, _ = bidirectional_dataset.create_combined_dataset(subjects=validation_subjects,
                                                                                proportion_to_use=1, time_before_cutout=time_before_cutout,
                                                                                                 cutout_duration=cutout_duration,
                                                                                original_freq=original_freq, resample_freq=resample_freq,
                                                                                standardization_info=standardization_info)
    
    # Create dataloader
    validation_loader = torch.utils.data.DataLoader(combined_validation_dataset, batch_size=batch_size, shuffle=False)
    # Run benchmark and get loss
    loss = run_benchmark(benchmark_model, loss_fn, validation_loader, device) 
    run[f"loss/{subjects[i]}"] = loss # Save loss to neptune
    run[f"n_sections/{subjects[i]}"] = len(individual_validation_dataset[0]) # save number of sections to neptune
    # Save loss and number of sections for calculating mean loss
    losses.append(loss)
    n_sections.append(len(individual_validation_dataset[0]))
    
total_sections = sum(n_sections) # Total number of sections for all subjects
run["n_sections/total"] = total_sections
# Calculate mean loss by weighing mean loss from each subject by proportion of total section
mean_loss = sum([losses[i] * n_sections[i] / total_sections for i in range(len(subjects))]) 

# Save mean loss to neptune
run["loss/total"] = mean_loss

run.stop() 